# Chapter 6 — RAG: Grounding Aegis in Your Runbooks

Aegis knows how to investigate. It does not know **your** incident-response runbooks or
the CVE advisory published last Tuesday. No model does, and no amount of prompting fixes
it — the knowledge has to be retrieved.

**Covered:** §6.1 the two-phase pipeline · §6.2 embeddings and similarity · §6.3.2 four
chunking strategies · §6.4.2 query transformation · §6.5.2 hybrid retrieval with BM25.

This chapter also has an optional **RAGAS** section at the end — the real library, in its
own runtime. See the warning there before installing it.


## Setup

Every lab in this book installs from **one** `requirements.txt` in the companion
repository. No notebook pins its own versions: change a dependency there and it
changes everywhere, including CI. That is how the labs mirror a production
service rather than a pile of scratch files.

The clone below fails loudly on purpose. A setup step that swallows its own
error surfaces later as a confusing `ModuleNotFoundError`, and you waste an hour
looking in the wrong place.


In [ ]:
REPO_URL = "https://github.com/<your-org>/<your-repo>.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


In [ ]:
!pip -q install -r requirements.txt


Now verify the environment before running any lab code. This is the same check CI
runs, and it catches the one dependency conflict that would otherwise waste your
afternoon.


In [ ]:
!python tools/check_env.py


### Choosing a model tier

The labs read `AEGIS_MODEL` and swap the model behind a single seam:

| Tier | Cost | Determinism | Use it for |
|---|---|---|---|
| `mock` | free, no key | identical every run | learning the control flow; the test suite; CI |
| `openai` | billed per call | varies run to run | seeing a real model make these decisions |

Start on `mock`. Everything in this chapter runs there. When you switch to
`openai`, the code does not change — only the seam does.

Set the key from the environment, never as a literal in a cell. In Colab use the
key icon in the sidebar (Secrets); the cell below reads it without printing it.


In [ ]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key

# To use a real model instead, uncomment these two lines:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: "); os.environ["AEGIS_MODEL"] = "openai"

print("model tier:", os.environ["AEGIS_MODEL"])


## §6.1 + §6.2 — Index, then retrieve

Two phases. Indexing happens once: split documents into chunks, embed each chunk.
Retrieval happens per query: embed the question, compare, return the closest.

Everything interesting is in the splitting.


In [ ]:
import sys
sys.path.insert(0, "ch06")
from rag.pipeline import chunk_fixed, embed, cosine
from data.corpus import all_docs

docs = dict(all_docs())
print("corpus:", list(docs))

# PHASE 1 - index
index = []
for doc_id, text in docs.items():
    for chunk in chunk_fixed(text, size=240):
        index.append({"doc": doc_id, "text": chunk, "vector": embed(chunk)})
print(f'indexed {len(index)} chunks from {len(docs)} documents')

# PHASE 2 - retrieve
query = "how do I contain an account takeover"
qv = embed(query)
ranked = sorted(index, key=lambda c: cosine(qv, c["vector"]), reverse=True)

print()
print(f'query: {query!r}')
top = ranked[0]
print(f'retrieved from: {top["doc"]}  similarity {cosine(qv, top["vector"]):.3f}')
print(f'  {top["text"][:90]}...')


## §6.3.2 — Four chunking strategies

Chunking decides what the retriever can *find*. Four strategies, trading precision
against context.

Watch the trap in the next cell: on a four-document corpus **every** strategy retrieves
the right runbook. Pass/fail tells you nothing. The signal is the confidence spread.


In [ ]:
from rag.pipeline import (STRATEGIES, chunk_fixed, chunk_sentence_window,
                          chunk_semantic, chunk_hierarchical)

doc = docs["rb_account_takeover"]
print("the four strategies, and what each optimizes for:")
print("  chunk_fixed             blind character windows - the baseline it usually loses to")
print("  chunk_sentence_window   a sentence plus neighbors - precise match, restored context")
print("  chunk_semantic          consecutive sentences grouped so a runbook step stays whole")
print("  chunk_hierarchical      index the child, return the parent section")
print()
print(f'{"strategy":18} {"chunks":>7} {"avg chars":>10}')
for name, fn in STRATEGIES.items():
    chunks = fn(doc)
    avg = sum(len(c) for c in chunks) / len(chunks)
    print(f'{name:18} {len(chunks):7} {avg:10.0f}')

print()
print("now retrieve with each, on the same query:")
qv = embed("how do I contain an account takeover")
scores = {}
for name, fn in STRATEGIES.items():
    best = max(cosine(qv, embed(c)) for c in fn(doc))
    scores[name] = best
    print(f'  {name:18} best-chunk similarity {best:.3f}')

spread = max(scores.values()) - min(scores.values())
print()
print(f'all four found the right document. the spread is {spread:.3f}.')
print("THAT is the signal: on 40,000 documents, hundreds of irrelevant chunks")
print("score above the weakest strategy by coincidence, and retrieval fails SILENTLY.")


## §6.4.2 — Query transformation

Users do not write queries the way documents are written. Rewriting the user's words
into the document's vocabulary is often a bigger win than any chunking change.


In [ ]:
from rag.hybrid import rewrite_query

for query in ["help my account got taken over", "someone hacked us"]:
    print("before:", query)
    print("after: ", rewrite_query(query))
    print()


## §6.5.2 — Hybrid retrieval: where dense search fails

Dense embeddings capture **meaning** and are bad at exact tokens — identifiers, error
codes, CVE numbers. A CVE id is exactly what an analyst searches for and exactly what an
embedding blurs.

Two advisories below are formulaic and near-identical, differing only in the identifier.
That is how CVE advisories actually read. Uses `rank_bm25`, the standard open-source
implementation.

One honest note: the book's toy `embed` is bag-of-words, so it would match the literal
token `2026-2000` and *hide* this failure. `make_semantic_embed` collapses identifiers
the way a real embedding model does — otherwise this section would assert a failure the
code quietly disproves.


In [ ]:
from rag.hybrid import HybridRetriever, make_semantic_embed

corpus = dict(docs)
corpus["cve_2026_2000"] = ("CVE-2026-2000 advisory. Severity high. Affected component: "
                           "VPN appliance. Mitigation: apply the vendor patch and rotate "
                           "credentials.")
corpus["cve_2026_1000"] = ("CVE-2026-1000 advisory. Severity high. Affected component: "
                           "VPN appliance. Mitigation: apply the vendor patch and rotate "
                           "credentials.")

retriever = HybridRetriever(corpus, make_semantic_embed(embed), cosine)
query = "CVE-2026-2000 mitigation"
print(f'query: {query!r}   (the analyst wants ONE specific advisory)')
print()

dense = sorted(retriever.dense(query).items(), key=lambda kv: -kv[1])[:2]
print("dense only  ->", [(d, round(s, 3)) for d, s in dense])
print(f'   dense cannot separate them: {abs(dense[0][1] - dense[1][1]) < 0.001}')

sparse = sorted(retriever.sparse(query).items(), key=lambda kv: -kv[1])[:2]
print("sparse only ->", [(d, round(s, 3)) for d, s in sparse])

fused = retriever.search(query, alpha=0.5)[:2]
print("hybrid      ->", [(r["doc"], r["fused"]) for r in fused])
print()
print("Dense returned the WRONG advisory first, at an identical score.")
print("BM25 separated them. Neither is better - they fail differently,")
print("which is the entire argument for running both.")


### The alpha dial

`alpha` weights dense against sparse. It is another measured dial that silently sets
quality, and it ships as a vendor default.


In [ ]:
print(f'{"alpha":>6}  top result')
for alpha in (0.0, 0.25, 0.5, 0.75, 1.0):
    top = retriever.search(query, alpha=alpha)[0]
    kind = "pure sparse" if alpha == 0 else "pure dense" if alpha == 1 else "hybrid"
    print(f'{alpha:>6}  {top["doc"]:16} ({kind})')


## Optional — real RAGAS

Everything above is retrieval. Measuring whether the *answers* are grounded is
evaluation, and RAGAS is the standard open-source library for it.

**Read this before installing.** RAGAS conflicts with the modern LangChain stack:
`ragas==0.4.3` imports `langchain_community.chat_models.vertexai`, which no longer
exists in `langchain-community` 0.4.x. Pip resolves it **cleanly** and then `import
ragas` fails at runtime. The book's `requirements.txt` pins
`langchain-community==0.3.29` defensively for exactly this reason.

Run the RAGAS cells in a **fresh runtime** (Runtime ▸ Restart). Chapter 10 does the full
evaluation treatment; here it is just enough to score the retrieval you built above.


In [ ]:
# Fresh runtime recommended. Installs from the book's single requirements.txt.
!pip -q install -r requirements.txt


In [ ]:
from ragas.metrics.collections import BleuScore, ExactMatch

# Deterministic RAGAS metrics: real library, no LLM, no API key, no cost.
bleu, exact = BleuScore(), ExactMatch()

REFERENCE = "Disable the affected account and revoke all active sessions."
for label, response in (("grounded  ", REFERENCE),
                        ("partial   ", "Disable the account and revoke sessions."),
                        ("fabricated", "Deploy the zero-trust mesh and rotate the HSMs.")):
    b = bleu.score(reference=REFERENCE, response=response)
    e = exact.score(reference=REFERENCE, response=response)
    print(f'{label}  BleuScore={b.value:.3f}   ExactMatch={e.value:.0f}')


---

## What you built

A two-phase RAG pipeline, four chunking strategies compared on confidence spread rather
than pass/fail, query rewriting, and hybrid retrieval that recovers the exact-identifier
case dense search loses.

- **All four strategies 'work' on a small corpus.** The spread is the signal.
- **Dense and sparse fail differently.** That is why you run both.
- **Chunk size, alpha, and the score threshold are dials** that silently set quality.

**Next:** Chapter 7 gives Aegis a plan, the ability to repair it, and the judgment to
refuse a conclusion the evidence does not support.
